# Joyory SkinGenie — Colab T4 training (HAM10000 derm classifier)

**Runtime → Change runtime type → T4 GPU** before running. All heavy training happens here, never on the laptop.

Goal: fine-tune EfficientNet-B0 on HAM10000 (7 classes: akiec, bcc, bkl, df, mel, nv, vasc) and export:
- `derm_ham10000.pt` (state dict + meta)
- `derm_ham10000.onnx` (**self-contained**, no external `.data` file)

Realistic target: single-model balanced accuracy ~0.80–0.85. This is a **cosmetic shopping-assistant signal, not a medical device**.

In [ ]:
import torch
print('cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
!nvidia-smi -L

In [ ]:
!pip -q install kagglehub scikit-learn pandas matplotlib seaborn huggingface_hub onnx onnxruntime

## 1. Download HAM10000 (via kagglehub — no Kaggle credentials needed)

In [ ]:
import kagglehub, glob, os
ds = kagglehub.dataset_download('kmader/skin-cancer-mnist-ham10000')
print(ds)
print(glob.glob(os.path.join(ds, '*')))
DATA = ds

## 2. Stratified split + loaders (augmentation on train only)

In [ ]:
import pandas as pd, numpy as np, os
from PIL import Image
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms

CLASSES = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
meta = pd.read_csv(os.path.join(DATA, 'HAM10000_metadata.csv'))
meta = meta[meta['dx'].isin(CLASSES)].reset_index(drop=True)
print(meta['dx'].value_counts())

img_dirs = [os.path.join(DATA, d) for d in ('HAM10000_images_part_1', 'HAM10000_images_part_2')]
def find_img(image_id):
    for d in img_dirs:
        for ext in ('.jpg', '.JPG', '.png'):
            p = os.path.join(d, image_id + ext)
            if os.path.exists(p):
                return p
    return None
meta['path'] = meta['image_id'].map(find_img)
meta = meta[meta['path'].notna()].reset_index(drop=True)
print('usable images:', len(meta))

train_df, val_df = train_test_split(meta, test_size=0.15, stratify=meta['dx'], random_state=42)
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_tfm = transforms.Compose([transforms.RandomResizedCrop(224, scale=(0.8, 1.0)), transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.1, 0.1, 0.1), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
val_tfm = transforms.Compose([transforms.Resize((256, 256)), transforms.CenterCrop(224),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

class HamDS(Dataset):
    def __init__(self, df, tfm):
        self.df, self.tfm = df.reset_index(drop=True), tfm
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        x = self.tfm(Image.open(r['path']).convert('RGB'))
        return x, CLASSES.index(r['dx'])

counts = train_df['dx'].value_counts()
w = train_df['dx'].map((1.0 / counts)).values
sampler = WeightedRandomSampler(w, len(w))
train_ld = DataLoader(HamDS(train_df, train_tfm), batch_size=64, sampler=sampler, num_workers=2)
val_ld = DataLoader(HamDS(val_df, val_tfm), batch_size=128, num_workers=2)
print('train/val:', len(train_df), len(val_df))

## 3. Model: ImageNet EfficientNet-B0 + 7-class head, class-weighted loss

In [ ]:
import torch.nn as nn
from torchvision import models
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = models.efficientnet_b0(weights='IMAGENET1K_V1')
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 7)
model.to(device)
freq = train_df['dx'].value_counts().reindex(CLASSES).values.astype(float)
cw = torch.tensor(freq.sum() / (7 * freq), dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=cw)
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=15)
scaler = torch.cuda.amp.GradScaler(enabled=(device == 'cuda'))
print('class weights:', cw.cpu().numpy().round(2))

## 4. Train (AMP on T4) — keep best balanced-accuracy checkpoint

In [ ]:
from sklearn.metrics import balanced_accuracy_score
best, best_state = 0.0, None
for epoch in range(15):
    model.train()
    for x, y in train_ld:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        with torch.cuda.amp.autocast(enabled=(device == 'cuda')):
            loss = criterion(model(x), y)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
    sched.step()
    model.eval()
    P, T = [], []
    with torch.no_grad():
        for x, y in val_ld:
            P += model(x.to(device)).argmax(1).cpu().tolist()
            T += y.tolist()
    bacc = balanced_accuracy_score(T, P)
    print(f'epoch {epoch+1:02d} val_bacc={bacc:.4f} lr={sched.get_last_lr()[0]:.2e}')
    if bacc > best:
        best, best_state = bacc, {k: v.cpu() for k, v in model.state_dict().items()}
print('best val_bacc:', round(best, 4))
torch.save({'state_dict': best_state, 'classes': CLASSES, 'val_bacc': best}, 'derm_ham10000.pt')

## 5. Evaluate: per-class report + confusion matrix

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt, seaborn as sns
model.load_state_dict(best_state); model.eval()
P, T = [], []
with torch.no_grad():
    for x, y in val_ld:
        P += model(x.to(device)).argmax(1).cpu().tolist()
        T += y.tolist()
print(classification_report(T, P, target_names=CLASSES, digits=3))
cm = confusion_matrix(T, P, normalize='true')
plt.figure(figsize=(7, 5)); sns.heatmap(cm, annot=True, fmt='.2f', xticklabels=CLASSES, yticklabels=CLASSES)
plt.title('Row-normalized confusion matrix'); plt.show()

## 6. Export self-contained ONNX (must have NO external `.data` file) + verify

In [ ]:
import torch, glob, os
model.load_state_dict(best_state); model.eval()
dummy = torch.randn(1, 3, 224, 224)
torch.onnx.export(model, dummy, 'derm_ham10000.onnx', input_names=['image'], output_names=['logits'],
                  opset_version=17, do_constant_folding=True)
print('onnx size KB:', os.path.getsize('derm_ham10000.onnx') // 1024)
print('external data files:', glob.glob('derm_ham10000.onnx.data'))
assert not glob.glob('derm_ham10000.onnx.data'), 'ONNX uses external data — re-export with smaller opset/folding'
import onnxruntime as ort, numpy as np
sess = ort.InferenceSession('derm_ham10000.onnx', providers=['CPUExecutionProvider'])
out = sess.run(None, {'image': np.random.rand(1, 3, 224, 224).astype(np.float32)})[0]
print('onnx logits shape:', out.shape, '-> 7 classes OK' if out.shape == (1, 7) else 'MISMATCH')

## 7. Download artifacts to the laptop (`backend/models/`) + optional Hub upload

In [ ]:
from google.colab import files
files.download('derm_ham10000.pt')
files.download('derm_ham10000.onnx')
# Optional: publish weights (uncomment + set HF_TOKEN)
# from huggingface_hub import HfApi
# HfApi().upload_file(path_or_fileobj='derm_ham10000.pt', path_in_repo='derm_ham10000.pt',
#                      repo_id='<user>/joyory-derm-ham10000', repo_type='model')

## 8. (Best-effort) Re-export a FIXED self-contained `skin_signals.onnx`

Upstream `skin_signals.onnx` references a missing external data file. If you download `skin_signals_best.pt`, rebuild the published architecture (EfficientNet-B0 backbone → shared 1280→512→256 → 4 sigmoid heads), load with `strict=False`, align keys, and export exactly like cell 6. Verify zero `.data` files before replacing `backend/models/skin_signals.onnx`.